Notebook is used to work with Deployment scripts using PowerShell and Azure CLI when needed.  If you would like to use Cloud Shell in Notebook see ref:  [Title](https://github.com/dotnet/interactive/blob/main/samples/notebooks/powershell/Docs/Interact-With-Azure-Cloud-Shell.ipynb)

In [1]:
#Login to Azure
#Use -Environment Parm to if you need to use Azure Gov.  AzureUSGovernmnet

Connect-AzAccount


To override which subscription Connect-AzAccount selects by default, use `Update-AzConfig -DefaultSubscriptionForLogin 00000000-0000-0000-0000-000000000000`. Go to https://go.microsoft.com/fwlink/?linkid=2200610 for more information.

Account                SubscriptionName TenantId                             Environment
-------                ---------------- --------                             -----------
frgarofa@microsoft.com DSE EDog         72f988bf-86f1-41af-91ab-2d7cd011db47 AzureCloud



In [2]:
#Select Subscription

#$subscription = "Azure FFL Internal Subscription FRGAROFA (GOV)"

$subscription = "Azure FFL Internal Subscription FRGAROFA"

Select-AzSubscription -Subscription $subscription



Name                                     Account        SubscriptionNa Environment    TenantId
                                                        me
----                                     -------        -------------- -----------    --------
Azure FFL Internal Subscription FRGAROF… frgarofa@micr… Azure FFL Int… AzureCloud     72f988bf-86f…



In [3]:
#Create Resource Group if not setup for data maangment zone:

$ResourceGroup = 'dmz-cloudscale-dev'

New-AzResourceGroup -Name $ResourceGroup -Location 'East US'


Confirm
Provided resource group already exists. Are you sure you want to update it?
[Y] Yes  [N] No  [S] Suspend  [?] Help(default is 'Y')

Error: Command cancelled.

In [3]:
#Check Resource Group 

$ResourceGroup = 'dmz-cloudscale-dev'

Get-AzResource -ResourceGroupName $ResourceGroup | Format-Table -AutoSize


Name                                                                               ResourceGroupNam
                                                                                   e
----                                                                               ----------------
purviewacct-5h7tsge77gt7i-dev                                                      dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-account-pep-dev.nic.880b6bb7-4341-4ff2-a6e9-1e3f7b32456e dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-portal-pep-dev.nic.6fc3c8f6-477e-4cc2-98db-e15f85500ef6  dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-account-pep-dev                                          dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-portal-pep-dev                                           dmz-cloudscale-…



In [8]:
#Set Param values for ARM deployment template

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\arm\DMZ\Purview\Purview.template.json'
    TemplateParameterFile = '..\arm\DMZ\Purview\Purview.parameters.json'
    Verbose = $true
}

In [9]:
#Test ARM deplkoyment
Test-AzResourceGroupDeployment @params



SubscriptionNotFound - The subscription '02cd7474-626e-4aa4-b21f-4856bdb92779' could not be found.



In [29]:
#Deploy ARM Template

New-AzResourceGroupDeployment @params

VERBOSE: Performing the operation "Creating Deployment" on target "dmz-cloudscale-dev".
VERBOSE: 2:21:28 PM - Template is valid.
VERBOSE: 2:21:29 PM - Create template deployment 'Purview.template'
VERBOSE: 2:21:29 PM - Checking deployment status in 5 seconds
New-AzResourceGroupDeployment: 
Line |
   3 |  New-AzResourceGroupDeployment @params
     |  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
     | 2:21:34 PM - The deployment 'Purview.template' failed with error(s). Showing 1 out of 1 error(s).
Status Message: The resource type could not be found in the namespace 'Microsoft.Purview' for api version '2020-12-01-preview'. (Code:InvalidResourceType)

CorrelationId: f5399086-5a79-45d2-a404-b52fde850995

DeploymentName          : Purview.template
ResourceGroupName       : dmz-cloudscale-dev
ProvisioningState       : Failed
Timestamp               : 7/19/2023 6:21:33 PM
Mode                    : Incremental
TemplateLink            : 
Parameters              : 
                          Name      

In [20]:
#Get list of Supported API versions for resource ProviderNamespace

Get-AzResourceProvider -ProviderNamespace Microsoft.Purview | Select-Object ProviderNamespace -ExpandProperty ResourceTypes | Select-Object * -ExpandProperty ApiVersions | Sort-Object -Descending

2021-12-01
2021-12-01
2021-12-01
2021-12-01
2021-12-01
2021-12-01
2021-07-01
2021-07-01
2021-07-01
2021-07-01
2021-07-01
2021-07-01
2020-12-01-preview
2020-12-01-preview
2020-12-01-preview
2020-12-01-preview
2020-12-01-preview
2020-12-01-preview


In [19]:
#Set default API version for Purview

$token = Get-AzAccessToken -Resource 'https://management.usgovcloudapi.net/'
$bearer = $token.Token

$params = @{
    Uri     = 'https://management.usgovcloudapi.net/subscriptions/02cd7474-626e-4aa4-b21f-4856bdb92779/providers/microsoft.purview/register?api-version=2021-07-01-preview'
    Method  = 'Post'
    Headers = @{'Authorization' = "Bearer $($bearer)" }
    Body    = '{"resourcetype": "Microsoft.Purview/accounts","location":["usgovvirginia"],"apiVersions": ["2021-12-01","2021-07-01","2020-12-01-preview"],"defaultApiVersion": "2019-06-01","capabilities": "None"
    }'
   }

Invoke-RestMethod @params



id                 : /subscriptions/02cd7474-626e-4aa4-b21f-4856bdb92779/providers/Microsoft.Purvie
                     w
namespace          : Microsoft.Purview
authorizations     : {@{applicationId=73c2949e-da2d-457a-9607-fcc665198967; 
                     roleDefinitionId=1BC09725-0C9B-4F57-A3D0-FCCF4EB40120; 
                     managedByRoleDefinitionId=9e3af657-a8ff-583c-a75c-2fe7c4bcb635}, 
                     @{applicationId=fd642066-7bfc-4b65-9463-6a08841c12f0; 
                     roleDefinitionId=5b3958cb-0439-42e1-80e3-cae077d0d5e8}}
resourceTypes      : {@{resourceType=operations; locations=System.Object[]; 
                     apiVersions=System.Object[]; capabilities=None}, 
                     @{resourceType=setDefaultAccount; locations=System.Object[]; 
                     apiVersions=System.Object[]; defaultApiVersion=2021-07-01; 
                     capabilities=None}, @{resourceType=removeDefaultAccount; 
                     locations=System.Object[]; apiV

In [3]:
#Test and Deploy with Bicep

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\bicep\DMZ\Purview\purview.bicep'
    Verbose = $true
}
#Test-AzResourceGroupDeployment @params

New-AzResourceGroupDeployment @params



VERBOSE: Using Bicep v0.16.1
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\Purview\purview.bicep(107,22) : Warning BCP081: Resource type "Microsoft.Purview/accounts@2021-12-01" does not have types available.
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\Purview\purview.bicep(122,24) : Warning BCP081: Resource type "Microsoft.Purview/accounts/kafkaConfigurations@2021-12-01" does not have types available.
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\Purview\purview.bicep(139,33) : Warning BCP081: Resource type "Microsoft.Network/privateEndpoints@2023-04-01" does not have types available.
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\Purview\purview.bicep(145,7) : Warning use-resource-id-functions: If property "id" represents a resource ID, it must use a symbolic resource reference, be a parameter or start with one of these functions: extensionResourceId, guid, if, reference, resourceId, subscription, subscriptionResourceId, tenantResourceId. [https://aka.ms/bicep/linter/use-resource-id-function